# Sleep intrusions during NOD and extended wake

Frequency and properties of sleep intrusions (brief NREM bouts) occurring during novel
object deprivation (NOD) and extended wake (EXT), comparing early against late periods
within each.

- `early_nod` / `late_nod`: first / last 1h cumulative of Wake + NREM during NOD
- `early_ext` / `late_ext`: first / last 1h cumulative of Wake + NREM during extended wake
- Sleep intrusions are NREM bouts within these hypnograms


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pubplots as pp

from cnpix_local_sleep import hyp as offhyp
from cnpix_local_sleep import sps_conf

## Load hypnograms


In [ ]:
CONDITIONS_OF_INTEREST = ["Early.NOD", "Late.NOD"]  # , "early_ext", "late_ext"]

subject_probes = sps_conf.get_subject_probe_list(
    method="morphological",
    exclude_thalamus=True,
    exclude_striatum=True,
    exclude_other=True,
)

# Load hypnograms for each unique (subject, probe), without renaming conditions.
# Use conditions=None to avoid filtering by renamed dot-notation keys.
all_hgs = {}
for subject, probe in subject_probes:
    if (subject, probe) in all_hgs:
        continue
    hgs = offhyp.load_statistical_condition_hypnograms(subject, probe)
    available = {c: hgs[c] for c in CONDITIONS_OF_INTEREST if c in hgs}
    if available:
        all_hgs[(subject, probe)] = available

print(f"Loaded hypnograms for {len(all_hgs)} subject-probe pairs")
for (subj, prb), hgs in all_hgs.items():
    print(f"  {subj} / {prb}: {list(hgs.keys())}")

## Sleep intrusion extraction


In [ ]:
def compute_intrusion_stats(
    hg,
) -> dict:
    """Compute sleep intrusion statistics from a condition hypnogram containing Wake and NREM bouts."""
    total_duration = hg["duration"].sum()
    intrusions = hg.keep_states(["NREM"])
    n = len(intrusions)
    durations = intrusions["duration"].values

    stats = {
        "n_intrusions": n,
        "total_condition_duration_s": total_duration,
        "total_intrusion_s": durations.sum() if n > 0 else 0.0,
        "frac_intrusion": durations.sum() / total_duration
        if (n > 0 and total_duration > 0)
        else 0.0,
        "mean_duration_s": durations.mean() if n > 0 else np.nan,
        "median_duration_s": np.median(durations) if n > 0 else np.nan,
        "std_duration_s": durations.std() if n > 1 else np.nan,
        "min_duration_s": durations.min() if n > 0 else np.nan,
        "max_duration_s": durations.max() if n > 0 else np.nan,
    }

    # Inter-intrusion intervals: gaps between consecutive NREM bouts
    if n > 1:
        starts = intrusions["start_time"].values
        ends = intrusions["end_time"].values
        intervals = starts[1:] - ends[:-1]
        stats["mean_inter_interval_s"] = intervals.mean()
        stats["median_inter_interval_s"] = np.median(intervals)
        stats["min_inter_interval_s"] = intervals.min()
        stats["max_inter_interval_s"] = intervals.max()
    else:
        stats["mean_inter_interval_s"] = np.nan
        stats["median_inter_interval_s"] = np.nan
        stats["min_inter_interval_s"] = np.nan
        stats["max_inter_interval_s"] = np.nan

    return stats

## Per-subject sleep intrusion table


In [ ]:
rows = []
for (subject, probe), hgs in all_hgs.items():
    for condition, hg in hgs.items():
        stats = compute_intrusion_stats(hg)
        rows.append(
            {"subject": subject, "probe": probe, "condition": condition, **stats}
        )

df_stats = pd.DataFrame(rows)
df_stats

## Aggregate summary statistics


In [ ]:
numeric_cols = [
    "n_intrusions",
    "total_intrusion_s",
    "frac_intrusion",
    "mean_duration_s",
    "median_duration_s",
    "mean_inter_interval_s",
]

for condition in CONDITIONS_OF_INTEREST:
    subset = df_stats[df_stats["condition"] == condition]
    n_subjects = len(subset)
    n_with_intrusions = (subset["n_intrusions"] > 0).sum()

    print(f"\n{'=' * 60}")
    print(f"Condition: {condition}")
    print(f"  Subjects: {n_subjects}")
    print(f"  Subjects with sleep intrusions: {n_with_intrusions} / {n_subjects}")
    print(
        f"  Subjects without sleep intrusions: {n_subjects - n_with_intrusions} / {n_subjects}"
    )
    print()

    for col in numeric_cols:
        values = subset[col].dropna()
        if len(values) > 0:
            mean = values.mean()
            sem = values.sem() if len(values) > 1 else np.nan
            print(f"  {col}: {mean:.2f} +/- {sem:.2f} (n={len(values)})")
        else:
            print(f"  {col}: no data")

## Statistical significance of the early -> late increase

Is the increase in the number (`n_intrusions`) and total duration (`total_intrusion_s`)
of sleep intrusions from `Early.NOD` to `Late.NOD` statistically significant?

The data are paired, since each subject-probe is measured in both conditions, and
pseudoreplicated, since some subjects contribute two probes. The primary test therefore
operates at the subject level: probes are averaged within subject, then a paired
Wilcoxon signed-rank test (non-parametric, robust to the many `Early.NOD` zeros) is run
across subjects, with a paired t-test alongside. A mixed model with a subject random
intercept then confirms the result while retaining all probes. This mirrors the
workspace pattern in
`cnpix_local_sleep.morphological.pipeline.cross_structure_offs.test_excess_above_chance`.


In [ ]:
# Subject-level paired significance tests (primary).
#
# The data are paired (each subject-probe is measured in both Early.NOD and
# Late.NOD) and pseudoreplicated (some subjects contribute two probes). Following
# the workspace pattern (cnpix_local_sleep.morphological.pipeline.cross_structure_offs
# .test_excess_above_chance), we test at the SUBJECT level: average each subject's
# probes to one value per condition, then run a paired test across subjects.
# Counts/durations are heavily non-normal (many Early.NOD zeros), so the Wilcoxon
# signed-rank test is the headline; a paired t-test is reported alongside.
from scipy import stats

test_metrics = ["n_intrusions", "total_intrusion_s", "frac_intrusion"]

# Collapse probes -> one value per (subject, condition).
per_subject = (
    df_stats.groupby(["subject", "condition"], observed=True)[test_metrics]
    .mean()
    .reset_index()
)

print(
    f"Paired across {per_subject['subject'].nunique()} subjects "
    f"(probes averaged within subject); Late.NOD vs Early.NOD\n"
)

for col in test_metrics:
    wide = per_subject.pivot(index="subject", columns="condition", values=col)
    wide = wide.dropna(subset=CONDITIONS_OF_INTEREST)
    early = wide["Early.NOD"].to_numpy()
    late = wide["Late.NOD"].to_numpy()
    n = len(wide)
    diff = late - early

    # Wilcoxon needs the directional ("greater" = Late > Early) and two-sided p.
    w_one = stats.wilcoxon(late, early, alternative="greater")
    w_two = stats.wilcoxon(late, early, alternative="two-sided")
    t_one = stats.ttest_rel(late, early, alternative="greater")

    print(f"{col}  (n = {n} subjects)")
    print(f"  Early.NOD: median {np.median(early):.3f}, mean {early.mean():.3f}")
    print(f"  Late.NOD : median {np.median(late):.3f}, mean {late.mean():.3f}")
    print(
        f"  paired difference (Late - Early): median {np.median(diff):+.3f}, "
        f"mean {diff.mean():+.3f}"
    )
    print(
        f"  Wilcoxon signed-rank: W = {w_one.statistic:.1f}, "
        f"p = {w_one.pvalue:.2e} (one-sided, increase), "
        f"p = {w_two.pvalue:.2e} (two-sided)"
    )
    print(
        f"  paired t-test: t({n - 1}) = {t_one.statistic:.2f}, "
        f"p = {t_one.pvalue:.2e} (one-sided, increase)\n"
    )


In [ ]:
# Mixed-model robustness check (probe-preserving).
#
# The paired tests above collapse each subject's probes to a single mean. To
# confirm that the increase is not an artifact of that collapsing, refit on the
# full per-probe table while absorbing within-subject clustering in a random
# intercept (subject as the grouping factor). This honors every probe without
# treating the two probes of a subject as independent replicates.
import statsmodels.formula.api as smf

print("Mixed model:  metric ~ condition + (1 | subject)")
print(
    f"  (Late.NOD vs Early.NOD; {df_stats['subject'].nunique()} subjects, "
    f"{len(df_stats)} subject-probe rows)\n"
)

for col in test_metrics:
    sub = df_stats[["subject", "condition", col]].dropna()
    # Order so the coefficient is the Late - Early effect.
    sub = sub.assign(
        condition=pd.Categorical(
            sub["condition"], categories=CONDITIONS_OF_INTEREST, ordered=True
        )
    )
    try:
        fit = smf.mixedlm(f"{col} ~ condition", sub, groups=sub["subject"]).fit()
        term = [t for t in fit.params.index if t.startswith("condition")][0]
        coef = float(fit.params[term])
        p_two = float(fit.pvalues[term])
        ci_lo, ci_hi = fit.conf_int().loc[term].tolist()
        # One-side toward the predicted increase (Late > Early).
        p_one = p_two / 2 if coef > 0 else 1 - p_two / 2
        print(f"{col}:")
        print(f"  Late - Early coef = {coef:+.3f}  [95% CI {ci_lo:+.3f}, {ci_hi:+.3f}]")
        print(
            f"  p (one-sided, increase) = {p_one:.2e}   p (two-sided) = {p_two:.2e}   "
            f"converged = {fit.converged}\n"
        )
    except Exception as exc:  # noqa: BLE001 -- report, don't crash the notebook
        print(f"{col}: mixed model failed -> {type(exc).__name__}: {exc}\n")


## Sleep intrusion count and duration comparison


In [ ]:
plot_metrics = [
    ("n_intrusions", "Number of sleep intrusions"),
    ("total_intrusion_s", "Total sleep intrusion time (s)"),
    ("frac_intrusion", "Fraction of time in sleep intrusion"),
]

# Pairs to connect with lines: early-to-late within each deprivation type
paired_conditions = [
    ("Early.NOD", "Late.NOD"),
    # ("early_ext", "late_ext"),
]
condition_positions = {c: i for i, c in enumerate(CONDITIONS_OF_INTEREST)}

with pp.destination("figma"):
    fig, axes = plt.subplots(
        1, len(plot_metrics), figsize=pp.scale(1 * len(plot_metrics), 2)
    )

    for ax, (col, ylabel) in zip(axes, plot_metrics):
        sns.stripplot(
            data=df_stats,
            x="condition",
            y=col,
            ax=ax,
            order=CONDITIONS_OF_INTEREST,
            jitter=True,
            alpha=0.7,
            size=7,
            color="k",
        )
        sns.boxplot(
            data=df_stats,
            x="condition",
            y=col,
            ax=ax,
            order=CONDITIONS_OF_INTEREST,
            showfliers=False,
            boxprops=dict(facecolor="none"),
            whiskerprops=dict(color="gray"),
            medianprops=dict(color="black"),
        )

        # Draw paired lines connecting early to late within each deprivation type
        paired = df_stats.pivot(
            index=["subject", "probe"], columns="condition", values=col
        )
        for cond_early, cond_late in paired_conditions:
            if cond_early in paired.columns and cond_late in paired.columns:
                paired_valid = paired.dropna(subset=[cond_early, cond_late])
                x0 = condition_positions[cond_early]
                x1 = condition_positions[cond_late]
                for _, row in paired_valid.iterrows():
                    ax.plot(
                        [x0, x1],
                        [row[cond_early], row[cond_late]],
                        color="gray",
                        alpha=0.3,
                        linewidth=0.8,
                    )

        ax.set_ylabel(ylabel)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)

    fig.suptitle("Sleep intrusions across conditions")
    fig.tight_layout()
    plt.show()
    fig.savefig(
        "./outputs/intrusion_analysis/sleep_intrusion_paired_condition_stats.svg"
    )

## Sleep intrusion bout duration distribution


In [ ]:
# Collect all individual sleep intrusion bout durations
bout_rows = []
for (subject, probe), hgs in all_hgs.items():
    for condition, hg in hgs.items():
        intrusions = hg.keep_states(["NREM"])
        for _, bout in intrusions._df.iterrows():
            bout_rows.append(
                {
                    "subject": subject,
                    "probe": probe,
                    "condition": condition,
                    "duration_s": bout["duration"],
                }
            )

df_bouts = pd.DataFrame(bout_rows)
print(f"Total sleep intrusion bouts: {len(df_bouts)}")
for cond in CONDITIONS_OF_INTEREST:
    n = (df_bouts["condition"] == cond).sum()
    print(f"  {cond}: {n} bouts")

In [ ]:
import cnpix_local_sleep.plots

pal = cnpix_local_sleep.plots.get_condition_palette()
show_median = False
if len(df_bouts) > 0:
    with pp.destination("figma"):
        fig, axes = plt.subplots(2, 2, figsize=pp.scale(4, 4), sharey=True)
        axes_flat = axes.flatten()

        for ax, condition in zip(axes_flat, CONDITIONS_OF_INTEREST):
            subset = df_bouts[
                (df_bouts["condition"] == condition) & (df_bouts["duration_s"] < 60)
            ]
            if len(subset) > 0:
                ax.hist(
                    subset["duration_s"],
                    bins=np.arange(0, 61, 1),
                    edgecolor="black",
                    color=pal["Late.NOD.Wake"],
                    alpha=0.7,
                )
                if show_median:
                    ax.axvline(
                        subset["duration_s"].median(),
                        color="red",
                        linestyle="--",
                        label=f"median = {subset['duration_s'].median():.1f}s",
                    )
                    ax.legend()
            ax.set_xlabel("Bout duration (s)")
            ax.set_ylabel("Count")
            ax.set_title(f"{condition} (n={len(subset)} bouts)")

        fig.suptitle("Sleep intrusion bout duration distribution", fontsize=14)
        fig.tight_layout()
        plt.show()
        fig.savefig(
            "./outputs/intrusion_analysis/sleep_intrusion_bout_duration_histograms.svg"
        )
else:
    print("No sleep intrusion bouts found.")

## Per-subject hypnogram detail view


In [ ]:
state_colors = {"Wake": "white", "NREM": "steelblue"}

subjects_sorted = sorted(all_hgs.keys(), key=lambda x: x[0])
n_subjects = len(subjects_sorted)
n_conditions = len(CONDITIONS_OF_INTEREST)

with pp.destination("figma"):
    fig, axes = plt.subplots(
        n_subjects,
        n_conditions,
        figsize=pp.scale(7 * n_conditions, 0.6 * n_subjects),
        squeeze=False,
        sharex=False,
    )

    for row_idx, (subject, probe) in enumerate(subjects_sorted):
        hgs = all_hgs[(subject, probe)]
        for col_idx, condition in enumerate(CONDITIONS_OF_INTEREST):
            ax = axes[row_idx, col_idx]
            if condition in hgs:
                hg = hgs[condition]
                # Normalize times to start at 0 for display
                t0 = hg["start_time"].min()
                for _, bout in hg._df.iterrows():
                    color = state_colors.get(bout["state"], "gray")
                    ax.axvspan(
                        bout["start_time"] - t0,
                        bout["end_time"] - t0,
                        facecolor=color,
                        edgecolor="black" if bout["state"] == "NREM" else "none",
                        linewidth=0.5,
                        alpha=0.8,
                    )
                ax.set_xlim(0, hg["end_time"].max() - t0)
            else:
                ax.text(
                    0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes
                )

            ax.set_yticks([])
            if col_idx == 0:
                ax.set_ylabel(
                    subject.split("-")[0],
                    fontsize=7,
                    rotation=0,
                    ha="right",
                    va="center",
                )
            if row_idx == 0:
                ax.set_title(condition, fontsize=10)
            if row_idx < n_subjects - 1:
                ax.set_xticks([])
            else:
                ax.set_xlabel("Time from start (s)", fontsize=8)

    fig.suptitle(
        "Per-subject hypnograms (NREM = blue, Wake = white)", fontsize=12, y=1.02
    )
    fig.tight_layout()
    plt.show()
    fig.savefig("./outputs/intrusion_analysis/per_subject_hypnograms.svg")